<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 02
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Khai thác và trực quan hóa dữ liệu bằng Tableau"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  TIỀN XỬ LÝ BỘ DỮ LIỆU TMDB MOVIES
</div>

Notebook này ghi lại bước tiền xử lý đầu tiên cho đồ án Power BI sử dụng dataset TMDB Movies. Nội dung hiện tại tập trung vào việc lọc các phim có quốc gia sản xuất thuộc 5 nước: Việt Nam, Thái Lan, Nhật Bản, Hàn Quốc và Trung Quốc. Việc lọc theo quốc gia giúp thu hẹp phạm vi dữ liệu phù hợp với đề tài so sánh đặc điểm, mức độ phổ biến và đánh giá của phim trong nhóm quốc gia được chọn.

Ở bước này, notebook chỉ thực hiện lọc dữ liệu theo quốc gia sản xuất. Các bước làm sạch sâu, chuẩn hóa kiểu dữ liệu và tách bảng fact/dimension sẽ được thực hiện sau.

## 1. Giới thiệu mục tiêu tiền xử lý

Dataset gốc là dữ liệu phim từ TMDB. Mục tiêu của bước tiền xử lý đầu tiên là lọc ra các phim có quốc gia sản xuất thuộc 5 nước: Việt Nam, Thái Lan, Nhật Bản, Hàn Quốc và Trung Quốc. Việc lọc này giúp giới hạn dữ liệu đúng với phạm vi phân tích của đồ án Power BI, đồng thời giữ nguyên dữ liệu gốc để các bước xử lý tiếp theo có thể được kiểm soát rõ ràng.

Notebook hiện tại chưa thực hiện các thao tác làm sạch sâu hoặc tách bảng quan hệ.

## 2. Import thư viện và cấu hình đường dẫn

Bước này nạp các thư viện cần dùng và khai báo đường dẫn dữ liệu. File CSV gốc được đọc từ thư mục `data/raw/`, còn kết quả sau khi lọc sẽ được lưu vào `data/processed/`. Nếu thư mục output chưa tồn tại, notebook sẽ tự tạo để không làm thay đổi file dữ liệu gốc.

In [ ]:
import ast
import json
import os
import re
from pathlib import Path

import pandas as pd

# Khai báo đường dẫn dữ liệu để dễ chỉnh sửa khi cần.
RAW_DATA_PATH = Path("data/raw/TMDB_movie_dataset.csv")
PROCESSED_DIR = Path("data/processed")

# Nếu chạy notebook từ thư mục notebooks/, tự điều chỉnh về thư mục gốc của project.
if not RAW_DATA_PATH.exists() and Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
    RAW_DATA_PATH = PROJECT_ROOT / RAW_DATA_PATH
    PROCESSED_DIR = PROJECT_ROOT / PROCESSED_DIR

OUTPUT_FILE_PATH = PROCESSED_DIR / "tmdb_asia_5_countries_filtered.csv"

# Tạo thư mục output nếu chưa tồn tại.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file: {RAW_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE_PATH}")

## 3. Đọc dữ liệu thô và kiểm tra tổng quan

Bước này đọc dữ liệu gốc để kiểm tra kích thước, danh sách cột và một vài dòng đầu tiên. Việc kiểm tra tổng quan giúp xác nhận dataset có cột `production_countries`, đây là cột chính được dùng để lọc quốc gia sản xuất trong bước tiếp theo. Nếu thiếu cột này, notebook sẽ báo lỗi rõ ràng để tránh xử lý sai dữ liệu.

In [ ]:
# Đọc dữ liệu gốc từ file CSV.
df = pd.read_csv(RAW_DATA_PATH)

print(f"Số dòng ban đầu: {df.shape[0]:,}")
print(f"Số cột ban đầu: {df.shape[1]:,}")

print("Danh sách cột:")
print(df.columns.tolist())

# Kiểm tra sự tồn tại của cột dùng để lọc quốc gia sản xuất.
required_column = "production_countries"
if required_column not in df.columns:
    raise ValueError(f"Dataset thiếu cột bắt buộc: {required_column}")

df.head()

## 4. Lọc phim theo quốc gia sản xuất

### 4.1. Lý do lọc theo `production_countries`

Notebook lọc theo `production_countries` thay vì `original_language` vì quốc gia sản xuất phản ánh phạm vi phân tích sát hơn. `original_language` chỉ thể hiện ngôn ngữ gốc của phim và có thể gây nhầm lẫn trong một số trường hợp. Ví dụ, một phim nói tiếng Trung có thể đến từ Trung Quốc, Hong Kong, Đài Loan hoặc khu vực khác. Vì vậy, ở bước đầu tiên này, dữ liệu chỉ được lọc theo quốc gia sản xuất.

Các quốc gia mục tiêu gồm Vietnam, Thailand, Japan, South Korea và China. Tạm thời notebook không lấy Hong Kong, Taiwan hoặc Macau.

### 4.2. Chuẩn hóa dữ liệu quốc gia sản xuất

Cột `production_countries` có thể xuất hiện dưới nhiều dạng như chuỗi văn bản thông thường, chuỗi dạng list, chuỗi JSON/dict/list hoặc giá trị rỗng. Vì vậy, bước này xây dựng hàm trích xuất tên quốc gia một cách linh hoạt, sau đó tạo thêm các cột hỗ trợ phân tích trong Power BI: `selected_countries`, `selected_countries_text` và `primary_selected_country`.

In [ ]:
TARGET_COUNTRIES = ["Vietnam", "Thailand", "Japan", "South Korea", "China"]
TARGET_COUNTRY_SET = set(TARGET_COUNTRIES)

def extract_country_names(value):
    """Trích xuất danh sách tên quốc gia từ nhiều định dạng dữ liệu khác nhau."""
    if pd.isna(value):
        return []

    if isinstance(value, list):
        items = value
    elif isinstance(value, dict):
        items = [value]
    else:
        text_value = str(value).strip()
        if not text_value:
            return []

        items = None
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed_value = parser(text_value)
                if isinstance(parsed_value, list):
                    items = parsed_value
                elif isinstance(parsed_value, dict):
                    items = [parsed_value]
                break
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                continue

        # Nếu không phải JSON/list/dict, xem dữ liệu là chuỗi quốc gia phân tách bằng dấu phẩy hoặc chấm phẩy.
        if items is None:
            items = re.split(r",|;", text_value)

    country_names = []
    for item in items:
        if isinstance(item, dict):
            name = item.get("name") or item.get("country") or item.get("english_name")
        else:
            name = str(item)

        if name and str(name).strip():
            country_names.append(str(name).strip())

    return country_names

def select_target_countries(value):
    """Giữ lại các quốc gia thuộc 5 nước mục tiêu theo thứ tự đã cấu hình."""
    countries = extract_country_names(value)
    countries_found = {country for country in countries if country in TARGET_COUNTRY_SET}
    return [country for country in TARGET_COUNTRIES if country in countries_found]

# Tạo các cột hỗ trợ lọc và đọc kết quả trong Power BI.
df_with_country_flags = df.copy()
df_with_country_flags["selected_countries"] = df_with_country_flags["production_countries"].apply(select_target_countries)
df_with_country_flags["selected_countries_text"] = df_with_country_flags["selected_countries"].apply(lambda countries: "; ".join(countries))
df_with_country_flags["primary_selected_country"] = df_with_country_flags["selected_countries_text"]

df_with_country_flags[["production_countries", "selected_countries_text", "primary_selected_country"]].head()

### 4.3. Thực hiện lọc dữ liệu

Sau khi xác định các quốc gia mục tiêu trong từng dòng, bước này tạo dataframe mới tên `filtered_df`. Notebook chỉ giữ các phim có ít nhất một quốc gia thuộc 5 nước mục tiêu và không ghi đè dataframe gốc.

In [ ]:
# Chỉ giữ các phim có ít nhất một quốc gia thuộc nhóm mục tiêu.
filtered_df = df_with_country_flags[df_with_country_flags["selected_countries"].str.len() > 0].copy()

print(f"Số dòng sau khi lọc: {filtered_df.shape[0]:,}")
print(f"Số cột sau khi lọc: {filtered_df.shape[1]:,}")

filtered_df.head()

## 5. Kiểm tra kết quả sau khi lọc

Sau khi lọc, cần kiểm tra số lượng dòng còn lại và phân bố phim theo từng quốc gia. Bước này giúp đảm bảo dữ liệu sau lọc đúng phạm vi đề tài. Vì một phim có thể thuộc nhiều quốc gia sản xuất, thống kê theo từng quốc gia sử dụng thao tác `explode` trên cột `selected_countries`.

In [ ]:
initial_row_count = len(df)
filtered_row_count = len(filtered_df)
remaining_ratio = filtered_row_count / initial_row_count if initial_row_count else 0

country_counts = (
    filtered_df.explode("selected_countries")
    .groupby("selected_countries")
    .size()
    .reindex(TARGET_COUNTRIES, fill_value=0)
    .rename("movie_count")
)

multi_country_movie_count = (filtered_df["selected_countries"].str.len() > 1).sum()

print(f"Tổng số dòng ban đầu: {initial_row_count:,}")
print(f"Tổng số dòng sau khi lọc: {filtered_row_count:,}")
print(f"Tỷ lệ dữ liệu còn lại sau lọc: {remaining_ratio:.2%}")
print(f"Số phim có nhiều hơn 1 quốc gia mục tiêu: {multi_country_movie_count:,}")

print("Số phim theo từng quốc gia mục tiêu:")
display(country_counts.reset_index().rename(columns={"selected_countries": "country"}))

preview_columns = [
    "id",
    "title",
    "release_date",
    "production_countries",
    "selected_countries_text",
    "primary_selected_country",
]

filtered_df[preview_columns].head(10)

## 6. Xuất dữ liệu đã lọc

Dữ liệu sau bước lọc được lưu thành file CSV để dùng cho các bước tiền xử lý tiếp theo. File này sẽ là đầu vào cho các bước làm sạch, chuẩn hóa và tách bảng quan hệ sau này. Việc xuất file chỉ áp dụng cho dataframe đã lọc, không làm thay đổi file dữ liệu gốc.

In [ ]:
# Xuất dữ liệu đã lọc ra file CSV trong thư mục processed.
filtered_df.to_csv(OUTPUT_FILE_PATH, index=False, encoding="utf-8-sig")

print(f"Đã lưu file: {OUTPUT_FILE_PATH}")
print(f"Shape của dữ liệu đã lưu: {filtered_df.shape}")

## 7. Các bước tiền xử lý dự kiến tiếp theo

Các bước dưới đây chỉ là kế hoạch cho giai đoạn tiếp theo, chưa được thực hiện trong notebook hiện tại:

- Kiểm tra và xử lý dữ liệu thiếu.
- Chuẩn hóa kiểu dữ liệu, đặc biệt là ngày phát hành, doanh thu, ngân sách và thời lượng.
- Xử lý các giá trị không hợp lệ như doanh thu hoặc ngân sách bằng 0 nếu cần.
- Tách cột `genres` thành bảng Thể loại và bảng Liên kết Phim - Thể loại.
- Tách `production_countries` thành bảng Quốc gia và bảng Liên kết Phim - Quốc gia.
- Tạo bảng Phim, bảng Hiệu suất phim và bảng Thời gian.
- Chuẩn bị dữ liệu cuối cùng để đưa vào Power BI.

Notebook hiện tại dừng ở bước lọc 5 quốc gia mục tiêu và xuất file filtered. Chưa tách bảng fact/dimension và chưa xóa các cột gốc quan trọng.